In [11]:
import sys, os
# add ../.. to path
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import control as ct
import numpy as np

import models.model_coef as TF
from models.mav_dynamics_control import MavDynamics
from models.compute_models import euler_state, quaternion_state
import parameters.control_parameters as AP
import parameters.aerosonde_parameters as MAV
import parameters.simulation_parameters as SIM
from message_types.msg_delta import MsgDelta


mav = MavDynamics(SIM.ts_simulation)
m = MAV.mass
g = MAV.gravity
u_star = AP.u_star
w_star = AP.w_star

x_trim = euler_state(TF.x_trim)
u_trim = TF.u_trim

ulast = x_trim.item(3)
vlast = x_trim.item(4)
wlast = x_trim.item(5)

def x_dot_euler(xdot_quat, x_euler, x_quat):
    # dTe_dxq (Tq(xe))
    # Te is euler_to_quaternion
    # Tq is quaternion_to_euler
    # Tq(xe) is quaternion_state(x_euler)
    # So dTe_dxq is Te(quaternion_state(x_euler) + eps - Te())
    dTe_dxq_of_xq = np.zeros((x_euler.size, x_quat.size))
    Te_out = x_euler
    n = x_quat.size # number of quaternion states
    eps = 0.001
    for i in range(n):
        x_quat_i = x_quat + eps * np.eye(n)[:, [i]]
        Te_outi = euler_state(x_quat_i)
        
        dTe_dquat_i = (Te_outi - Te_out) / eps
        dTe_dxq_of_xq[:, [i]] = dTe_dquat_i

    xdot_euler = dTe_dxq_of_xq @ xdot_quat
    return xdot_euler

# Define your nonlinear system using NonlinearIOSystem
def uparms_update(t, x, u, params):
    # Inputs come in 1D shape, convert to col vectors
    x_euler = x.reshape((-1,1))
    x_quat = quaternion_state(x_euler)
    u = u.reshape((-1,1))
    # jacobian functions need input to be in MsgDelta format which includes 2 additional unused inputs
    delta = MsgDelta()
    delta.from_array(np.pad(u, ((0,2),(0,0)), constant_values=0))
    wind = np.zeros((6, 1))
    
    # MAV dynamics are based on quaternion format
    mav._state = quaternion_state(x_quat)
    mav._update_velocity_data(wind)
    forces_moments = mav._forces_moments(delta)
    xdot_quat = mav._f(x_quat, forces_moments)
    
    xdot_euler = x_dot_euler(xdot_quat, x_euler, x_quat)

    return xdot_euler.flatten()


def uparms_output(t, x, u, params):
    global ulast, vlast, wlast
    theta = x.item(7)
    g = MAV.gravity
    dt = SIM.ts_simulation
    h = x.item(2)
    u = x.item(3)
    v = x.item(4)
    w = x.item(5)
    udot = (u - ulast) / dt
    vdot = (v - vlast) / dt
    wdot = (w - wlast) / dt
    
    ulast = u
    vlast = v
    wlast = w
    
    Vgdot = (u*udot + v*vdot + w*wdot) / np.sqrt(u**2 + v**2 + w**2)
    
    Edot = Vgdot / g + np.sin(theta)
    Ldot = Vgdot / g - np.sin(theta)
    return np.array([Edot, Ldot])

# Instantiate system
sys_nl = ct.NonlinearIOSystem(uparms_update, uparms_output, states=12, inputs=4, outputs=2, dt=SIM.ts_simulation)

# Linearize at operating point (State: [0,0], Input: [0])
x_eq = x_trim.flatten()
u_eq = u_trim.flatten()
sys_lin = ct.linearize(sys_nl, x_eq, u_eq)

# print("Linearized A matrix:\n", sys_lin.A)
# print("Linearized B matrix:\n", sys_lin.B)
A = sys_lin.A
B = sys_lin.B
C = sys_lin.C
D = sys_lin.D

E1 = np.array([
    [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    [0, 0, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
])
E2 = np.array([
    [1, 0, 0, 0],
    [0, 0, 0, 1]
])
E3 = np.array([
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0,	 0, 0, 0, 1, 0, 0, 0]
])
E4 = np.array([
    [0, 1, 0, 0],
    [0, 0, 1, 0]
])

A_lon = E1 @ A @ E1.T
B_lon = E1 @ B @ E2.T
C_lon = C @ E1.T
D_lon = D @ E2.T

A_lat = E3 @ A @ E3.T
B_lat = E3 @ B @ E4.T
C_lat = C @ E3.T
D_lat = D @ E4.T

print(f"A_lon =\n{A_lon}")
print(f"B_lon =\n{B_lon}")
print(f"C_lon =\n{C_lon}")
print(f"D_lon =\n{D_lon}")

A_lon =
[[-0.2822  0.4946 -1.242   0.      0.    ]
 [-0.5611 -4.4978 24.9691  0.1025  0.    ]
 [ 0.1986 -3.993   0.      0.      0.    ]
 [ 0.      0.      1.0001  0.      0.    ]
 [ 0.0497 -0.9988  0.     25.      0.    ]]
B_lon =
[[ -0.1392   9.4813]
 [ -2.5861   0.    ]
 [-36.1124   0.    ]
 [  0.       0.    ]
 [  0.       0.    ]]
C_lon =
[[10.1811  0.5064  0.      0.9988  0.    ]
 [10.1811  0.5064  0.     -0.9988  0.    ]]
D_lon =
[[0. 0.]
 [0. 0.]]


In [ ]:
import numpy as np
from numpy import array, concatenate, zeros, eye


"""
Get longitudinal trim statespace linearization
States x = [u, w, q, theta, h]
Inputs u = [de, dt]
"""
new_states = 2 # Edot_int, Ldot_int
# States x = [u, w, q, theta, h, Edot_int, Ldot_int]
Ahat = concatenate((
    concatenate((A_lon, zeros((A_lon.shape[0], new_states))), axis=1),
    concatenate((C_lon, zeros((C_lon.shape[0], new_states))), axis=1)),
    axis=0
)

Bhat = concatenate((B_lon, 
                    D_lon), 
                    axis=0)

Chat = concatenate((
    concatenate((C_lon, zeros((2,2))), axis=1),
    concatenate((zeros((2, C_lon.shape[1])), eye(2,2)), axis=1)),
                axis=0)
Dhat = concatenate((D_lon,
                    zeros((2,2))),
                    axis=0)

# Pretty Print Ahat and Bhat
np.set_printoptions(suppress=True, precision=4, linewidth=120)
print("Ahat =")
print(Ahat)
print("\nBhat =")
print(Bhat)
print("\nChat =")
print(Chat)
print("\nDhat =")
print(Dhat)

Ahat =
[[-0.2822  0.4946 -1.242   0.      0.      0.      0.    ]
 [-0.5611 -4.4978 24.9691  0.1025  0.      0.      0.    ]
 [ 0.1986 -3.993   0.      0.      0.      0.      0.    ]
 [ 0.      0.      1.0001  0.      0.      0.      0.    ]
 [ 0.0497 -0.9988  0.     25.      0.      0.      0.    ]
 [10.1811  0.5064  0.      0.9988  0.      0.      0.    ]
 [10.1811  0.5064  0.     -0.9988  0.      0.      0.    ]]

Bhat =
[[ -0.1392   9.4813]
 [ -2.5861   0.    ]
 [-36.1124   0.    ]
 [  0.       0.    ]
 [  0.       0.    ]
 [  0.       0.    ]
 [  0.       0.    ]]

Chat =
[[10.1811  0.5064  0.      0.9988  0.      0.      0.    ]
 [10.1811  0.5064  0.     -0.9988  0.      0.      0.    ]
 [ 0.      0.      0.      0.      0.      1.      0.    ]
 [ 0.      0.      0.      0.      0.      0.      1.    ]]

Dhat =
[[0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]]


In [23]:
from numpy import diag
from scipy.linalg import solve_continuous_are, inv

Qlon = diag([10.0, 100.0, 100, 10, 50.0, 60.0, 50.0]) # u, w, q, theta, h, Edot_int, Ldot_int
Rlon = diag ([1 , 1]) # e , t
Plon = solve_continuous_are(Ahat, Bhat, Qlon, Rlon)
Klon = inv(Rlon) @ Bhat.T @ Plon

print(f"Klon =\n{Klon}")

Klon =
[[   0.0316   -3.1912  -10.4104 -153.572    -7.0584   -0.2484    0.3577]
 [   5.6717    0.1272   -0.0393   -0.9609    0.0739    5.7199    4.7677]]
